# Comparative filtering with morphological trees

This tutorial compares a max-tree, min-tree, Tree of Shapes, unrestricted residual tree, and saturated residual tree. All five receive the same controlled image and the same increasing area criterion, exposing how hierarchy representation changes reconstruction.

## Goal

By the end, you will be able to:

- build all five families through the public Python API;
- compute node support area;
- apply `filteringByPruningMin` at an area threshold;
- distinguish polar max/min filtering from self-dual filtering;
- inspect the saturation restriction on residual trees;
- compare filter rules for non-monotone BitQuad descriptors;
- run an Ultimate Attribute Opening assisted by classical MSER.

## Setup

The installed package is imported directly with `import mmcfilters`. Prepare the optional notebook dependencies outside the notebook by following `README.md`.

In [ ]:
import mmcfilters
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

print(f"mmcfilters: {mmcfilters.__version__}")


### Key assumptions

- The image is two-dimensional, C-contiguous, and `uint8`.
- Component and residual trees use radius 1.5 (8-connectivity).
- The Tree of Shapes uses the default self-dual producer policy.
- Linear pixel 0 is the infinity reference for the saturated residual tree.
- Minimum-pruning keeps nodes with area $A(n) \geq \lambda$.
- Equal numeric thresholds need not represent equal granulometries because each tree organizes supports differently.
- Median BitQuad thresholds are demonstrations, not optimized segmentation parameters.
- Classical MSER is demonstrated separately on a smooth-level max-tree because it requires global altitude order and a gray-level neighbourhood.

## Steps

### 1. Create a controlled image

The image contains nested bright and dark levels plus two $2	imes2$ noise patches. Alternating inclusions make the difference between accepting every current regional extremum and accepting only saturated extrema visible.

In [ ]:
ROWS = COLS = 64
row_grid, col_grid = np.ogrid[:ROWS, :COLS]
squared_distance = (row_grid - 32) ** 2 + (col_grid - 32) ** 2

image = np.full((ROWS, COLS), 128, dtype=np.uint8)
image[squared_distance <= 20**2] = 220
image[squared_distance <= 15**2] = 40
image[squared_distance <= 9**2] = 210
image[squared_distance <= 4**2] = 30

# Small extrema of opposite polarities.
image[5:7, 5:7] = 250
image[56:58, 56:58] = 5
image = np.ascontiguousarray(image)

fig, ax = plt.subplots(figsize=(5.2, 4.4), constrained_layout=True)
view = ax.imshow(image, cmap="gray", vmin=0, vmax=255)
ax.set_title("Input image")
ax.set_axis_off()
fig.colorbar(view, ax=ax, label="Gray level", shrink=0.82)
plt.show()


### 2. Build five hierarchies

| Family | Organized sets | Polarity | Role in this example |
| --- | --- | --- | --- |
| Max-tree | upper-level components | bright | removes small bright structures |
| Min-tree | lower-level components | dark | removes small dark structures |
| Tree of Shapes | upper and lower shapes | self-dual | handles both polarities in one inclusion tree |
| Unrestricted residual | synchronized current extrema | self-dual | accepts every current regional extremum |
| Saturated residual | saturated subset of current extrema | self-dual | requires the complement to stay connected to the chosen infinity pixel |

Saturation depends on domain topology and `infinityPixel`; it is not a threshold substitution.

In [ ]:
factory = mmcfilters.MorphologicalTreeFactory
ADJACENCY_RADIUS = 1.5
INFINITY_PIXEL = 0

trees = {
    "Max-tree": factory.createMaxTree(image, radius=ADJACENCY_RADIUS),
    "Min-tree": factory.createMinTree(image, radius=ADJACENCY_RADIUS),
    "Tree of Shapes": factory.createTreeOfShapes(image),
    "Unrestricted residual": factory.createSelfDualResidualTree(
        image,
        radius=ADJACENCY_RADIUS,
    ),
    "Saturated residual": factory.createSaturatedSelfDualResidualTree(
        image,
        infinityPixel=INFINITY_PIXEL,
        radius=ADJACENCY_RADIUS,
    ),
}


def enum_name(value) -> str:
    return str(value).split(".")[-1]


tree_summary = pd.DataFrame(
    [
        {
            "tree": name,
            "live nodes": tree.numNodes,
            "root": tree.getRoot(),
            "descriptive kind": enum_name(tree.descriptiveKind),
            "altitude order": enum_name(tree.altitudeOrder),
            "adjacency": enum_name(tree.adjacencyMode),
        }
        for name, tree in trees.items()
    ]
).set_index("tree")

display(tree_summary)

`INCREASING_FROM_ROOT` and `DECREASING_FROM_ROOT` express the global polarity of component trees. `UNCONSTRAINED` is expected for self-dual hierarchies, whose gray-level altitudes may alternate along a branch.

### 3. Compute area and apply pruning

Area is topological support cardinality. We use $\lambda=12$ to remove only four-pixel extrema and $\lambda=800$ to expose structural differences among hierarchies.

In [ ]:
AREA = mmcfilters.Attribute.AREA
AREA_THRESHOLDS = (12, 800)

area_by_tree = {
    name: mmcfilters.Attribute.computeSingleTopologyAttribute(
        tree,
        AREA,
        dtype=np.float64,
    )
    for name, tree in trees.items()
}

filtered_images = {}
result_rows = []

for name, tree in trees.items():
    filters = mmcfilters.AttributeFilters(tree)
    for threshold in AREA_THRESHOLDS:
        filtered = filters.filteringByPruningMin(
            area_by_tree[name],
            threshold,
        )
        filtered_images[(name, threshold)] = filtered
        result_rows.append(
            {
                "tree": name,
                "area threshold": threshold,
                "changed pixels": int(np.count_nonzero(filtered != image)),
                "remaining levels": int(np.unique(filtered).size),
                "minimum": int(filtered.min()),
                "maximum": int(filtered.max()),
            }
        )

filtering_summary = pd.DataFrame(result_rows).set_index(
    ["tree", "area threshold"]
)
display(filtering_summary)


In [ ]:
fig, axes = plt.subplots(
    len(AREA_THRESHOLDS),
    len(trees),
    figsize=(16, 6.2),
    constrained_layout=True,
)

for row, threshold in enumerate(AREA_THRESHOLDS):
    for col, name in enumerate(trees):
        axes[row, col].imshow(
            filtered_images[(name, threshold)],
            cmap="gray",
            vmin=0,
            vmax=255,
        )
        axes[row, col].set_title(
            f"{name}\n" + rf"$\lambda={threshold}$"
        )
        axes[row, col].set_axis_off()

fig.suptitle("Minimum pruning by AREA", fontsize=15)
plt.show()

At $\lambda=12$, the max-tree removes bright noise, the min-tree removes dark noise, and the self-dual hierarchies remove both. At $\lambda=800$, the organization of nested levels dominates reconstruction.

### 4. Inspect the threshold profile

The plot is not a quality metric. It counts pixels changed from the input and reveals each hierarchy's transition points.

In [ ]:
profile_thresholds = np.array([1, 5, 12, 50, 100, 300, 600, 800, 1300], dtype=float)
profile_rows = []

for name, tree in trees.items():
    filters = mmcfilters.AttributeFilters(tree)
    for threshold in profile_thresholds:
        filtered = filters.filteringByPruningMin(area_by_tree[name], threshold)
        profile_rows.append(
            {
                "tree": name,
                "threshold": threshold,
                "changed pixels": np.count_nonzero(filtered != image),
            }
        )

filtering_profile = pd.DataFrame(profile_rows)

fig, ax = plt.subplots(figsize=(9.6, 5.2), constrained_layout=True)
for name, group in filtering_profile.groupby("tree", sort=False):
    ax.plot(
        group["threshold"],
        group["changed pixels"],
        marker="o",
        linewidth=2,
        label=name,
    )

ax.set_title("Reconstruction sensitivity to area threshold")
ax.set_xlabel(r"Area threshold $\lambda$")
ax.set_ylabel("Changed pixels")
ax.grid(alpha=0.25)
ax.legend(ncol=2)
plt.show()

### 5. Filter with non-monotone BitQuad attributes

`BITQUADS_AREA` grows with support. Other BitQuad descriptors—topology, perimeter, and shape—are not guaranteed to increase along inclusion, so rejecting an ancestor does not imply rejecting all descendants.

For each descriptor, the demonstration threshold is the median over live nodes. Four rules are compared:

- **direct:** every accepted node recovers its own altitude;
- **subtractive:** only residuals of accepted nodes are accumulated;
- **maximum pruning:** a subtree is cut only when no descendant restores it;
- **Viterbi:** dynamic programming selects a connected preserved set.

The Tree of Shapes makes the comparison self-dual. `computeSingleAttribute(...)` lets the BitQuad backend select the appropriate directional adjacency from altitude context.

In [ ]:
bitquad_tree = trees["Unrestricted residual"]
bitquad_alive_nodes = np.asarray(bitquad_tree.getAliveNodeIds(), dtype=int)

bitquad_descriptors = {
    "BITQUADS_NUMBER_EULER": mmcfilters.Attribute.BITQUADS_NUMBER_EULER,
    "BITQUADS_NUMBER_HOLES": mmcfilters.Attribute.BITQUADS_NUMBER_HOLES,
    "BITQUADS_PERIMETER": mmcfilters.Attribute.BITQUADS_PERIMETER,
    "BITQUADS_PERIMETER_CONTINUOUS": mmcfilters.Attribute.BITQUADS_PERIMETER_CONTINUOUS,
    "BITQUADS_CIRCULARITY": mmcfilters.Attribute.BITQUADS_CIRCULARITY,
    "BITQUADS_PERIMETER_AVERAGE": mmcfilters.Attribute.BITQUADS_PERIMETER_AVERAGE,
    "BITQUADS_LENGTH_AVERAGE": mmcfilters.Attribute.BITQUADS_LENGTH_AVERAGE,
    "BITQUADS_WIDTH_AVERAGE": mmcfilters.Attribute.BITQUADS_WIDTH_AVERAGE,
}

bitquad_values = {}
bitquad_thresholds = {}
bitquad_outputs = {}
bitquad_rows = []

for descriptor_name, descriptor in bitquad_descriptors.items():
    values = mmcfilters.Attribute.computeSingleAttribute(
        bitquad_tree,
        descriptor,
        dtype=np.float64,
    )
    live_values = values[bitquad_alive_nodes]
    threshold = float(np.median(live_values))
    keep_criterion = (values > threshold).tolist()
    filters = mmcfilters.AttributeFilters(bitquad_tree)

    outputs = {
        "Direct": filters.filteringDirectRule(keep_criterion),
        "Subtractive": filters.filteringSubtractiveRule(keep_criterion),
        "Maximum pruning": filters.filteringByPruningMax(values, threshold),
        "Viterbi": filters.filteringByViterbiRule(values, threshold),
    }

    bitquad_values[descriptor_name] = values
    bitquad_thresholds[descriptor_name] = threshold
    for rule_name, filtered in outputs.items():
        bitquad_outputs[(descriptor_name, rule_name)] = filtered

    bitquad_rows.append(
        {
            "attribute": descriptor_name,
            "median threshold": threshold,
            "nodes above threshold": int(np.count_nonzero(values[bitquad_alive_nodes] > threshold)),
            **{
                f"{rule_name} — changed pixels": int(np.count_nonzero(filtered != image))
                for rule_name, filtered in outputs.items()
            },
        }
    )

bitquad_summary = pd.DataFrame(bitquad_rows).set_index("attribute")
display(bitquad_summary.round(4))

reference_descriptor = "BITQUADS_CIRCULARITY"
rule_names = ("Direct", "Subtractive", "Maximum pruning", "Viterbi")
fig, axes = plt.subplots(1, 5, figsize=(16.5, 3.4), constrained_layout=True)
axes[0].imshow(image, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Input")
axes[0].set_axis_off()

for axis, rule_name in zip(axes[1:], rule_names):
    axis.imshow(
        bitquad_outputs[(reference_descriptor, rule_name)],
        cmap="gray",
        vmin=0,
        vmax=255,
    )
    axis.set_title(rule_name)
    axis.set_axis_off()

fig.suptitle(
    "BITQUADS_CIRCULARITY — "
    f"median threshold = {bitquad_thresholds[reference_descriptor]:.3f}",
    fontsize=14,
)
plt.show()